In [1]:
import os

import duckdb
from dotenv import find_dotenv, load_dotenv

conn = duckdb.connect()

load_dotenv(find_dotenv())

True

In [2]:
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD")

In [ ]:
# Install and load spatial extension
conn.execute("INSTALL spatial; LOAD spatial;")

In [3]:
# for S3/MinIO access
conn.execute("INSTALL httpfs")
conn.execute("LOAD httpfs")

#  MinIO connection
conn.execute("SET s3_region = 'us-east-1'")  # Region doesn't matter here
conn.execute(f"SET s3_endpoint = '{MINIO_ENDPOINT}'")
conn.execute(f"SET s3_access_key_id = '{MINIO_ACCESS_KEY}'")
conn.execute(f"SET s3_secret_access_key = '{MINIO_SECRET_KEY}'")
conn.execute("SET s3_use_ssl = false")
conn.execute("SET s3_url_style = 'path'")  # Important for MinIO!

In [4]:
result = conn.execute("""
    SELECT
        id, name
    FROM read_parquet('s3://lakehouse-processed/communes_france.parquet')
    WHERE flag_sous_prefecture = 1
    AND id LIKE '09%'
""").df()

print(result)

      id          name
0  09225       Pamiers
1  09261  Saint-Girons


In [5]:
french_population_file = "s3://lakehouse-processed/french_population.parquet"
communes_france_file = "s3://lakehouse-processed/communes_france.parquet"

In [6]:
result = conn.execute(f"""
    SELECT
        id, name
    FROM read_parquet('{communes_france_file}')
    WHERE flag_sous_prefecture = 1
    AND id LIKE '09%'
""").df()

print(result)

      id          name
0  09225       Pamiers
1  09261  Saint-Girons


In [11]:
result = conn.execute(f"""
    SELECT
        com.id,
        name,
        area_m2 / 1e6 AS area_km2,
        population,
        population/area_km2 AS population_density,

    FROM read_parquet('{communes_france_file}') com
        LEFT JOIN read_parquet('{french_population_file}') pop
            ON com.id = pop.id
            AND pop.year = 2023
    WHERE flag_metropole = 1
    ORDER BY population_density asc
    LIMIT 50
""").df()

result

,id,name,area_km2,population,population_density
0,55189,Fleury-devant-Douaumont,24.612581,0,0.000000
1,55050,Bezonvaux,22.026559,0,0.000000
2,55139,Cumières-le-Mort-Homme,14.605688,0,0.000000
3,55307,Louvemont-Côte-du-Poivre,19.922125,0,0.000000
4,55039,Beaumont-en-Verdunois,18.604792,0,0.000000
5,55239,Haumont-près-Samogneux,26.083645,0,0.000000
6,26274,Rochefourchat,27.953488,2,0.071547
7,04107,Majastres,63.902618,5,0.078244
8,83147,Vérignon,77.305928,8,0.103485
9,57682,Turquestein-Blancrupt,68.679727,11,0.160164


In [ ]:
result = conn.execute(f"""
    SELECT distinct department_code
    FROM read_parquet('{communes_france_file}')
    order by 1
    LIMIT 1000
""").df()

print(result)